# Workshop 4 - Sistema RAG con NVIDIA NIM + ChromaDB

## Paso 1: Instalar dependencias

In [1]:
%pip install langchain langchain-nvidia-ai-endpoints langchain-huggingface langchain-community chromadb sentence-transformers python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## Paso 2: Fase A - Ingesta y creación del Vector Store

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

loader = TextLoader("docs/intro-to-llms-karpathy.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

persist_directory = "db/karpathy_chroma"
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=persist_directory
)
vector_store.persist()
print(f"✅ Vector store creado con {len(docs)} chunks.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Vector store creado con 81 chunks.


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  warn_deprecated(


## Paso 3: Fase B - Pipeline RAG

In [2]:
from dotenv import load_dotenv
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
api_key = os.environ.get("API_KEY")

llm = ChatNVIDIA(
    model="meta/llama-4-maverick-17b-128e-instruct",
    api_key=api_key,
    temperature=0.1
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma(
    persist_directory="db/karpathy_chroma",
    embedding_function=embeddings
)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

Context:
{context}

Question: {input}
""")

combine_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(vector_store.as_retriever(), combine_chain)

# Prueba rápida
result = qa_chain.invoke({"input": "What is retrieval augmented generation?"})
print(result["answer"])

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:486: UserWarning: Found meta/llama-4-maverick-17b-128e-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

According to the provided context, "retrieval augmented generation" is a process where a large language model (like ChatGPT) can reference chunks of text in uploaded files and use that information when creating responses. It's like browsing, but instead of browsing the internet, the model browses the uploaded files and uses them as reference information.


## Paso 4: Generar respuestas a las 50 preguntas

In [3]:
import json

with open("docs/questions.json", "r", encoding="utf-8") as f:
    questions_data = json.load(f)

questions = [item["question"] for item in questions_data]

json_results = []
for i, question in enumerate(questions):
    print(f"[{i+1}/{len(questions)}] {question[:60]}...")
    response = qa_chain.invoke({"input": question})
    json_results.append({
        "question": question,
        "answer": response["answer"],
        "contexts": [doc.page_content for doc in response["context"]]
    })

with open("my_rag_output.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(json_results, indent=4, ensure_ascii=False))

print(f"\n✅ Guardado my_rag_output.json con {len(json_results)} respuestas.")

[1/50] What are some security challenges associated with large lang...
[2/50] What is the purpose of the base model in the process of deve...
[3/50] What is an adversarial example in the context of large langu...
[4/50] What are the challenges associated with large language model...
[5/50] What are the key components of large language models and how...
[6/50] What is an adversarial example in the context of large langu...
[7/50] What is the significance of parameters and weights in the fu...
[8/50] What methods does the language model use for data collection...
[9/50] What is the significance of Meta AI in the development of la...
[10/50] What is the purpose of a universal transferable suffix in th...
[11/50] What is the concept of jailbreaking the model in the context...
[12/50] What are some capabilities of ChatGPT in handling complex qu...
[13/50] What is the significance of self-improvement in the developm...
[14/50] What is the process of fine tuning in the development of lan...
[

## Paso 5: Verificar resultados

In [6]:
with open("my_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[1], indent=1, ensure_ascii=False))

Total de preguntas respondidas: 50

--- Ejemplo (primera entrada) ---
{
 "question": "What is the purpose of the base model in the process of developing an assistant model?",
 "answer": "The base model is not directly usable because it doesn't answer questions with answers; it is essentially an internet document sampler. However, its purpose is that it has already undergone the expensive pre-training stage, making it a useful starting point for further fine-tuning to develop an assistant model.",
 "contexts": [
  "an assistant model, because we don't actually really just want document generators, that's not very helpful for many tasks, we want um to give questions to something and we wanted to generate answers based on those questions, so we really want an assistant model instead, and the way you obtain these assistant models is fundamentally through the following process, we basically keep the optimization identical, so the training will be the same, it's just the next word prediction